In [ ]:
import sys
sys.path.append('./pyten/')

import numpy as np
import pandas as pd
import pyten.tenclass
from FATR_Newton import FATR_Newton

In [ ]:
def prepare_data_from_matrix(rating_matrix, user_genders, feature_d=0):
    """
    Convert a full matrix 2D complete (users x features) in tensor 3D format for FATR_Newton
    
    :param rating_matrix: numpy array (n_users, n_features) with ratings
    :param user_genders: numpy array (n_users,)
    :param feature_d: index of the dimension of the tensor containing sensitive features (default 0 = users)
    :return: all necessary inputs for FATR_Newton function
    """
    
    n_users, n_features = rating_matrix.shape
    unique_genders = np.unique(user_genders, return_counts=True)
    n_genders = len(unique_genders[0])
    
    print "="*50
    print "PREPARAZIONE DATI"
    print "="*50
    print "Rating matrix shape:", rating_matrix.shape
    print "User genders shape:", user_genders.shape
    print "Number of users:", n_users
    print "Number of features:", n_features
    print "Matrix min/max:", np.min(rating_matrix), np.max(rating_matrix)
    
    print "\nGender distribution:"
    print "Unique genders:", unique_genders
    
    # Reshape 2d matrix into 3 dimensions (users x features x 1)
    # FATR_Netwon function accepts 3d tensor
    tensor_shape = (n_users, n_features, 1)
    rating_tensor = rating_matrix.reshape(tensor_shape)
    
    print "\nTensor shape:", tensor_shape
    
    # Create omega mask: We set all values to 1 since we suppose a full matrix without missing entries
    omega = np.ones(tensor_shape)
    print "All entries observed (omega=1):", np.sum(omega)
    
    # Create features for rows (sensitive feature is associated with rows)
    #feature_d = 0  # dimensione 0 = users
    feature_n = n_genders
    
    print("num genders: ", n_genders)
    # One-hot encoding of sensitive feature
    features = np.zeros((n_users, n_genders))
    for i in range(n_users):
        gender = int(user_genders[i])
        features[i, gender] = 1
    
    # Create omega_groups (mask for protected groups)
    omega_groups = []

    print "\nFeatures (gender encoding):"
    print "  Shape:", features.shape
    
    for group in range(n_genders):
        print "  Group {0}: {1} entries".format(group, np.sum(features[:, group]))
        mask_group = np.zeros(tensor_shape)
        group_user_indices = np.where(user_genders == group)[0]
        for user_idx in group_user_indices:
            mask_group[user_idx, :, :] = omega[user_idx, :, :]
        omega_groups.append(mask_group)
        print "\nGroup {0}: {1} entries".format(group, int(np.sum(mask_group)))
    
    # Check protected groups
    for i, mask in enumerate(omega_groups):
        if np.sum(mask) == 0:
            raise ValueError("The group {0} has no observed entries".format(i))
    
    # Convert to pyten tensor
    y = pyten.tenclass.Tensor(rating_tensor)
    
    return y, features, feature_d, feature_n, omega, omega_groups


def main_from_matrix(y, features, feature_d, feature_n, omega, omega_groups,
                     rating_matrix, user_genders, r=10, reg_para=0.1, 
                     Freg_para=0.1, tol=1e-4, maxiter=500, printitn=10):
    """
    Generate fair and recovered matrices from the original matrix using FATR_Newton function

    :param y: pyten tensor with data
    :param features: numpy array (n_users, n_genders) sensitive feature vector (one-hot encoded)
    :param feature_d: The mode of the tensor that the sensitie features belong to
    :param feature_n: The size of the sensitive feature vector
    :param omega: numpy array (n_users, n_features, 1) mask with all 1 (all data is observed)
    :param omega_groups: list of numpy arrays (n_users, n_features, 1) masks for each protected group
    :param rating_matrix: numpy array (n_users, n_features) data matrix
    :param user_genders: numpy array (n_users,) sensitive attribute associated with rows
    :param r: rank for factorization
    :param reg_para: regularization parameter
    :param Freg_para: regularization parameter for the Frobenius norm
    :param tol: tolerance on difference in fit
    :param maxiter: maximum number of iterations
    :param printitn: print fit every n iterations; 0 for no printing

    :return: fair_matrix, recovered_matrix, U, Uf, diff_fair, diff_recovered
    """
    
    #'U'  - factorized matrices
    #'Uf' - factorized matrices for fair tensor
    #'Xf' - recovered fair tensor
    #'X'  - recovered tensor
    U, Uf, X, Xf = FATR_Newton(
        y=y,
        features=features,
        feature_d=feature_d,
        feature_n=feature_n,
        r=r,
        omega=omega,
        omega_groups=omega_groups,
        reg_para=reg_para,
        Freg_para=Freg_para,
        tol=tol,
        maxiter=maxiter,
        init='random',
        printitn=printitn
    )
    
    fair_matrix = Xf.data.reshape(rating_matrix.shape)
    recovered_matrix = X.data.reshape(rating_matrix.shape)
    
    # frobenius error between original matrix and recovered (fair/standard) matrices
    diff_fair = np.linalg.norm(fair_matrix - rating_matrix)
    diff_recovered = np.linalg.norm(recovered_matrix - rating_matrix)
    
    print "\nDifferenza dalla matrice originale:"
    print "  Fair matrix (Frobenius norm):", diff_fair
    print "  Recovered matrix (Frobenius norm):", diff_recovered

    return fair_matrix, recovered_matrix, U, Uf, diff_fair, diff_recovered

In [ ]:
from fairness_metrics import balance_chierichetti, balance_gen, KL_fairness_error
from sklearn.metrics.cluster import normalized_mutual_info_score
from sklearn.metrics.cluster import adjusted_rand_score
from sklearn.metrics.cluster import adjusted_mutual_info_score
import compute_taus
import os 
import time

def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path)
        print("Folder created successfully.")
    else:
        print("Folder already exists.")

"""
DATASET = "movielens-1m"
#SENSITIVE = "gender"
SENSITIVE = "age"
TRUE_LABEL = "genres"
TRUE_LABEL_DIM = "cols"
"""

"""
DATASET = "amazon"
SENSITIVE = "gender"
TRUE_LABEL = "preferred_words_by_category"
TRUE_LABEL_DIM = "cols"
"""

"""
DATASET = "yelp"
SENSITIVE = "gender"
TRUE_LABEL = "restaurant_type"
TRUE_LABEL_DIM = "cols"
"""


DATASET = "lfw"
SENSITIVE = "gender"
TRUE_LABEL = "person_ids"
TRUE_LABEL_DIM = "rows"


if "movielens" in DATASET:
    PATH_DATASET = "./datasets/movielens/" + DATASET
else:
    PATH_DATASET = "./datasets/" + DATASET

PATH_RESULTS = "./results"
create_path(PATH_RESULTS)
PATH_RESULTS += "/" + DATASET
create_path(PATH_RESULTS)

if "movielens" in DATASET:
    PATH_RESULTS +=  "/" + SENSITIVE
    create_path(PATH_RESULTS)

matrix = np.load(PATH_DATASET + "/matrix.npy")
sensitive_feature = np.load(PATH_DATASET + "/" + SENSITIVE + ".npy")
true_labels = np.load(PATH_DATASET + "/" + TRUE_LABEL + ".npy")

results = {
    "run": [],
    "row_clus": [],
    "col_clus": [],
    "tau_x": [],
    "tau_y": [],
    "NMI_true_labels": [],
    "AMI_true_labels": [],
    "ARI_true_labels": [],
    "NMI_rows": [],
    "AMI_rows": [],
    "ARI_rows": [],
    "NMI_cols": [],
    "AMI_cols": [],
    "ARI_cols": [],
    "reconstruction_error": [],
    "balance_chierichetti": [],
    "balance_bera": [],
    "KL_fairness_error": [],
    "time": []
}

# Definisci l'ordine esplicitamente
results_keys = [
    "run",
    "row_clus",
    "col_clus",
    "tau_x",
    "tau_y",
    "NMI_true_labels",
    "AMI_true_labels",
    "ARI_true_labels",
    "NMI_rows",
    "AMI_rows",
    "ARI_rows",
    "NMI_cols",
    "AMI_cols",
    "ARI_cols",
    "reconstruction_error",
    "balance_chierichetti",
    "balance_bera",
    "KL_fairness_error",
    "time"
]
num_keys = len(results_keys)

if not os.path.exists(PATH_RESULTS + "/results_runs.csv"):
    with open(PATH_RESULTS + "/results_runs.csv", "a") as file:
        for idx, key in enumerate(results_keys):
            file.write("{}".format(key))
            if idx == num_keys - 1:
                file.write("\n")
            else:
                file.write(",")


vanilla_results = {
    "run": [],
    "row_clus": [],
    "col_clus": [],
    "tau_x": [],
    "tau_y": [],
    "NMI_true_labels": [],
    "AMI_true_labels": [],
    "ARI_true_labels": [],
    "reconstruction_error": [],
    "balance_chierichetti": [],
    "balance_bera": [],
    "KL_fairness_error": [],
    "time": []
}

# Definisci l'ordine esplicitamente
vanilla_results_keys = [
    "run",
    "row_clus",
    "col_clus",
    "tau_x",
    "tau_y",
    "NMI_true_labels",
    "AMI_true_labels",
    "ARI_true_labels",
    "reconstruction_error_recovered",
    "balance_chierichetti",
    "balance_bera",
    "KL_fairness_error",
    "time"
]
vanilla_num_keys = len(vanilla_results_keys)

if not os.path.exists(PATH_RESULTS + "/results_runs_vanilla.csv"):
    with open(PATH_RESULTS + "/results_runs_vanilla.csv", "a") as file:
        for idx, key in enumerate(vanilla_results_keys):
            file.write("{}".format(key))
            if idx == vanilla_num_keys - 1:
                file.write("\n")
            else:
                file.write(",")

y, features, feature_d, feature_n, omega, omega_groups = \
        prepare_data_from_matrix(matrix, sensitive_feature)

RUNS = 10

for run in range(RUNS):

    print("RUN {0}".format(run+1))

    start_time = time.time()
    fair_matrix, recovered_matrix, U_vanilla, U, diff_fair, diff_recovered = main_from_matrix(
        y, features, feature_d, feature_n, omega, omega_groups,
        rating_matrix=matrix,
        user_genders=sensitive_feature,
        r=10,              # rank del tensor
        reg_para=0.00001,      # regolarizzazione fairness (default)
        Freg_para=0.01,     # regolarizzazione Frobenius
        maxiter=500,       # iterazioni massime
        printitn=10        # stampa ogni 10 iterazioni
    )
    end_time = time.time()
    exec_time = end_time - start_time

    U_vanilla_users = U_vanilla[0]
    U_vanilla_features = U_vanilla[1]
    row_clus_vanilla = compute_taus.relabel_consecutive(np.argmax(U_vanilla_users, axis=1))
    col_clus_vanilla = compute_taus.relabel_consecutive(np.argmax(U_vanilla_features, axis=1))

    U_users = U[0]
    U_features = U[1]
    row_clus = compute_taus.relabel_consecutive(np.argmax(U_users, axis=1))
    col_clus = compute_taus.relabel_consecutive(np.argmax(U_features, axis=1))

    num_row_clusters = len(np.unique(row_clus))
    num_col_clusters = len(np.unique(col_clus))

    num_row_clusters_vanilla = len(np.unique(row_clus_vanilla))
    num_col_clusters_vanilla = len(np.unique(col_clus_vanilla))

    bera = balance_gen(sensitive_feature, row_clus)
    chierichetti = balance_chierichetti(sensitive_feature, row_clus)
    kl_fair = KL_fairness_error(row_clus, num_row_clusters, sensitive_feature)

    bera_vanilla = balance_gen(sensitive_feature, row_clus_vanilla)
    chierichetti_vanilla = balance_chierichetti(sensitive_feature, row_clus_vanilla)
    kl_fair_vanilla = KL_fairness_error(row_clus_vanilla, num_row_clusters_vanilla, sensitive_feature)

    if TRUE_LABEL_DIM == 'rows':
        NMI = normalized_mutual_info_score(true_labels, row_clus)
        ARI = adjusted_rand_score(true_labels, row_clus)
        AMI = adjusted_mutual_info_score(true_labels, row_clus)

        NMI_vanilla = normalized_mutual_info_score(true_labels, row_clus_vanilla)
        ARI_vanilla = adjusted_rand_score(true_labels, row_clus_vanilla)
        AMI_vanilla = adjusted_mutual_info_score(true_labels, row_clus_vanilla)
    else:
        NMI = normalized_mutual_info_score(true_labels, col_clus)
        ARI = adjusted_rand_score(true_labels, col_clus)
        AMI = adjusted_mutual_info_score(true_labels, col_clus)

        NMI_vanilla = normalized_mutual_info_score(true_labels, col_clus_vanilla)
        ARI_vanilla = adjusted_rand_score(true_labels, col_clus_vanilla)
        AMI_vanilla = adjusted_mutual_info_score(true_labels, col_clus_vanilla)

    NMI_rows = normalized_mutual_info_score(row_clus_vanilla, row_clus)
    NMI_cols = normalized_mutual_info_score(col_clus_vanilla, col_clus)
    AMI_rows = adjusted_mutual_info_score(row_clus_vanilla, row_clus)
    AMI_cols = adjusted_mutual_info_score(col_clus_vanilla, col_clus)
    ARI_rows = adjusted_rand_score(row_clus_vanilla, row_clus)
    ARI_cols = adjusted_rand_score(col_clus_vanilla, col_clus)

    X_norm = matrix/np.sum(matrix)
    tau_x, tau_y, _, _ = compute_taus.compute_taus(X_norm, 0, row_clus, col_clus)
    tau_x_vanilla, tau_y_vanilla, _, _ = compute_taus.compute_taus(X_norm, 0, row_clus_vanilla, col_clus_vanilla)

    """
    print(row_clus)
    print("bera: {0}".format(bera))
    print("chierichetti: {0}".format(chierichetti))
    print("KL fairness error: {0}".format(kl_fair))
    print("NMI: {0}".format(NMI))
    print("AMI: {0}".format(AMI))
    print("ARI: {0}".format(ARI))
    print("tau_x: {0}".format(tau_x))
    print("tau_y: {0}".format(tau_y))
    """

    with open(PATH_RESULTS + "/results_runs.csv", "a") as file:
        file.write("{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                   run,
                   num_row_clusters,
                   num_col_clusters,
                   tau_x,
                   tau_y,
                   NMI,
                   AMI,
                   ARI,
                   NMI_rows,
                   AMI_rows,
                   ARI_rows,
                   NMI_cols,
                   AMI_cols,
                   ARI_cols,
                   diff_fair,
                   diff_recovered,
                   chierichetti,
                   bera,
                   kl_fair,
                   exec_time
                   ))
        
    with open(PATH_RESULTS + "/results_runs_vanilla.csv", "a") as file:
        file.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                    run,
                    num_row_clusters_vanilla,
                    num_col_clusters_vanilla,
                    tau_x_vanilla,
                    tau_y_vanilla,
                    NMI_vanilla,
                    AMI_vanilla,
                    ARI_vanilla,
                    diff_recovered,
                    chierichetti_vanilla,
                    bera_vanilla,
                    kl_fair_vanilla,
                    exec_time
                    ))

    print "--- FINE ---"


In [ ]:
import numpy as np
print(np.__version__)

In [ ]:
restaurant_type = np.load('./data/yelp/restaurant_type.npy')
normalized_mutual_info_score(restaurant_type, features_cluster)

In [ ]:
Xf = np.load('fair_matrix.npy')
X = np.load('recovered_matrix.npy')

In [ ]:
np.matmul(U_users, U_features.T)